# Predicting High Recorded-Crime Risk in London MSOAs using 2021 Census Features

**中文翻译：** 使用 2021 年人口普查特征预测伦敦 MSOA 的高记录犯罪风险。


## Preparation

- Number of words: **English word count: 1,410** excluding code, code comments, tables, figures and references.
- Coding environment: SDS Docker / Anaconda / local Jupyter environment, with standard data science libraries.
- License: Creative Commons Attribution license.

**中文翻译：** 本节记录提交信息和正文词数。英文正文目前约 1,410 词，不包括代码、代码注释、表格、图和参考文献。




## Table of contents

1. [Introduction](#Introduction)

1. [Research questions](#Research-questions)

1. [Data](#Data)

1. [Methodology](#Methodology)

1. [Results and discussion](#Results-and-discussion)

1. [Conclusion](#Conclusion)

1. [References](#References)


## Introduction

[[ go back to the top ]](#Table-of-contents)

**English**

Recorded crime is not spread evenly across a city. Some neighbourhoods have much higher rates than others, and these differences matter for policing, local services and residents' everyday sense of safety. A large body of criminological work argues that crime is linked to both social conditions and place-based opportunity. Bursik and Grasmick (1993) connect neighbourhood crime to local social organisation and community control. Sampson, Raudenbush and Earls (1997) show that collective efficacy and concentrated disadvantage help explain differences in violence between urban neighbourhoods. Environmental criminology also argues that crime is shaped by routine activity, land use and the physical structure of urban space; Brantingham and Brantingham (1993) describe crime as patterned rather than random.

This project follows that place-based view of crime. It does not try to predict individual offenders or single crime events. Instead, it asks whether the social and housing characteristics of small areas can distinguish places with relatively high recorded-crime risk. My first attempt used ward-level crime and ward profile data, because wards are familiar policy units. However, the newer Metropolitan Police ward crime codes did not align well with the older ward profile geography. This created a serious matching problem and risked making the analysis depend on inconsistent boundaries. I therefore changed the design to use MSOAs, which provide a more consistent scale across Census 2021 features, LSOA-to-MSOA lookup data and MSOA boundaries.

**中文翻译**

记录犯罪在城市中的分布并不均匀。有些社区的犯罪率明显更高，这会影响警务资源、地方服务和居民的日常安全感。犯罪学研究通常认为，犯罪既与社会条件有关，也与地点中的机会结构有关。Bursik and Grasmick (1993) 将社区犯罪与地方社会组织和社区控制联系起来。Sampson, Raudenbush and Earls (1997) 说明 collective efficacy 和集中劣势可以解释城市社区之间的暴力差异。环境犯罪学也认为，犯罪受到日常活动、土地利用和城市空间结构的影响；Brantingham and Brantingham (1993) 认为犯罪具有空间模式，而不是随机分布。

本项目采用这种以地点为核心的犯罪理解。它不预测个体罪犯或单个犯罪事件，而是考察小区域的社会和住房特征能否区分相对高记录犯罪风险的地区。我最初尝试使用 ward 层级犯罪数据和 ward profile 数据，因为 ward 是常见的政策单元。但是，较新的 Metropolitan Police ward 犯罪编码和较旧的 ward profile 地理边界并不能很好匹配。这造成了严重的匹配问题，也会让分析依赖不一致的边界。因此我将设计改为使用 MSOA，因为 MSOA 能在 Census 2021 特征、LSOA-to-MSOA lookup 和 MSOA 边界之间提供更一致的尺度。




In [17]:
# 准备工作 / Setup: import libraries and set a fixed random seed for reproducibility.
# 环境提示 / Environment note: if packages are missing, install them in a separate cell.
# %pip install pandas numpy matplotlib seaborn scikit-learn geopandas shapely

import sqlite3
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import quote
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 图表风格 / Plot style: use consistent styling for clearer coursework figures.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RANDOM_STATE = 42
CRIME_YEAR = "2021"
CRIME_MONTHS = [f"{CRIME_YEAR}{month:02d}" for month in range(1, 13)]

# 数据来源 / Data source: read all datasets from GitHub raw URLs for reproducibility.
GITHUB_OWNER = "hou1020"
GITHUB_REPO = "CASA0006-coursework"
GITHUB_BRANCH = "main"
GITHUB_RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}"

def github_raw_url(path):
    return f"{GITHUB_RAW_BASE}/{quote(path, safe='/')}"

lsoa_crime_url = github_raw_url("MPS LSOA Level Crime (most recent 24 months).csv")
lsoa_historical_crime_url = github_raw_url("MPS LSOA Level Crime (Historical).csv")
oa_lsoa_msoa_lookup_url = github_raw_url("OA21_LAD22_LSOA21_MSOA21_LEP22_EN_LU_V2_789783291718145016.gpkg")
msoa_boundary_url = github_raw_url("Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3_-1334546435986816930.gpkg")

census_urls = {
    "age": github_raw_url("census2021/census2021-ts007a/census2021-ts007a-msoa.csv"),
    "deprivation": github_raw_url("census2021/census2021-ts011/census2021-ts011-msoa.csv"),
    "accommodation": github_raw_url("census2021/census2021-ts044/census2021-ts044-msoa.csv"),
    "tenure": github_raw_url("census2021/census2021-ts054/census2021-ts054-msoa.csv"),
    "economic": github_raw_url("census2021/census2021-ts066/census2021-ts066-msoa.csv"),
    "qualification": github_raw_url("census2021/census2021-ts067/census2021-ts067-msoa.csv"),
}


## Research questions

[[ go back to the top ]](#Table-of-contents)

**English**

The main research question is: **To what extent can 2021 Census socio-demographic and housing characteristics distinguish high recorded-crime risk MSOAs from other London MSOAs?**

The outcome is a binary label. An MSOA is labelled as high risk if its 2021 recorded-crime rate is in the top 25% of London MSOAs. The analysis compares a simple and interpretable Logistic Regression model with a more flexible Random Forest classifier. Because the high-risk class is the policy-relevant minority class, recall for high-risk MSOAs is treated as especially important.

**中文翻译**

本研究的主要问题是：**2021 年人口普查中的社会人口和住房特征，在多大程度上能够区分伦敦的高记录犯罪风险 MSOA 和其他 MSOA？**

因变量是一个二分类标签。如果某个 MSOA 的 2021 年记录犯罪率位于伦敦 MSOA 的前 25%，就被标记为高风险。本研究比较一个简单且可解释的 Logistic Regression 模型和一个更灵活的 Random Forest 分类器。由于高风险区域是政策上更重要的少数类，因此 high-risk recall 是特别重要的评价指标。


In [18]:
# 研究任务 / ML task linked to the research question:
# 使用 2021 Census MSOA 特征预测高犯罪风险 / Use 2021 Census MSOA features to classify high recorded-crime risk.
# 标签定义 / Label definition: y=1 for top-quartile 2021 MSOA crime-rate areas; y=0 otherwise.

research_question = (
    "To what extent can 2021 Census socio-demographic and built environment features "
    "distinguish high recorded-crime risk MSOAs from other London MSOAs?"
)
print(research_question)


To what extent can 2021 Census socio-demographic and built environment features distinguish high recorded-crime risk MSOAs from other London MSOAs?


## Data

[[ go back to the top ]](#Table-of-contents)

**English**

The main crime dataset is the Metropolitan Police Service LSOA-level recorded crime table. I use the twelve months from January 2021 to December 2021 so that the crime outcome is aligned with the 2021 Census. The original crime table is at LSOA level, while the explanatory variables are built at MSOA level. To connect the two geographies, I use the ONS OA/LSOA/MSOA lookup. Crime counts are first summed across offence categories within each LSOA, then mapped to MSOAs and summed again to produce an annual MSOA crime count.

The denominator for the crime rate is the Census 2021 usual resident population. The target variable is therefore the annual recorded-crime rate per 1,000 residents. This rate is more appropriate than a raw count because MSOAs have different population sizes. It also follows the assessment guidance, which recommends rates when the aim is to compare risk across places. The analysis uses 1,001 London MSOAs. In the data preparation checks, all 4,988 LSOAs in the MPS data are mapped to an MSOA, and all 1,001 crime MSOAs are matched to Census features.

The explanatory variables are selected from six Census 2021 MSOA tables: age, household deprivation, accommodation type, tenure, economic activity and qualifications. I keep a small set of interpretable variables, such as age structure, deprivation, flat housing, rented tenure, unemployment and education. ID columns, names and the LSOA aggregation count are used only for joins, checks or display, not as predictors. This choice reduces leakage and keeps the modelling task focused on meaningful area characteristics.

**中文翻译**

主要犯罪数据来自 Metropolitan Police Service 的 LSOA 层级记录犯罪表。本研究使用 2021 年 1 月到 12 月的数据，使犯罪结果和 2021 Census 在年份上保持一致。原始犯罪表是 LSOA 层级，而解释变量是 MSOA 层级。因此，本研究使用 ONS OA/LSOA/MSOA lookup 连接两个地理尺度。犯罪记录先在每个 LSOA 内按犯罪类型求和，再映射到 MSOA，并再次求和得到 MSOA 年犯罪数。

犯罪率的分母是 2021 Census 的常住人口。因此，目标变量是每 1,000 名居民的年度记录犯罪率。相比原始犯罪数，犯罪率更合适，因为不同 MSOA 的人口规模不同。这也符合评分要求中关于跨地区比较时应优先使用 rate 的建议。本研究包含 1,001 个伦敦 MSOA。数据检查显示，MPS 数据中的 4,988 个 LSOA 都成功映射到 MSOA，1,001 个 crime MSOA 也全部匹配到 Census 特征。

解释变量来自六个 Census 2021 MSOA 表：年龄、家庭贫困、住房类型、tenure、经济活动和学历。本研究保留少量可解释变量，例如年龄结构、贫困、flat housing、租房比例、失业和教育。编码、名称字段和 LSOA 聚合数量只用于连接、检查或展示，不作为模型预测变量。这样可以减少信息泄漏，并使建模任务集中在有意义的区域特征上。



In [19]:
# 1. 数据融合 / Data fusion: read LSOA crime, map it to MSOA, and merge Census features.
# 目标变量说明 / Target note: do not use ward-profile crime data or ward geography.
# Crime: MPS LSOA 2021 monthly counts -> aggregate to MSOA via OA/LSOA/MSOA lookup.
# Features: Census 2021 MSOA tables.
lsoa_crime_raw = pd.read_csv(lsoa_historical_crime_url)
print("LSOA crime data shape:", lsoa_crime_raw.shape)

month_cols = sorted([col for col in lsoa_crime_raw.columns if str(col).isdigit()])
annual_month_cols = CRIME_MONTHS
missing_months = [col for col in annual_month_cols if col not in lsoa_crime_raw.columns]
if missing_months:
    raise ValueError(f"Missing requested crime months in source data: {missing_months}")
print(f"Number of monthly crime columns in source: {len(month_cols)}")
print("Source date range:", min(month_cols), "to", max(month_cols))
print("Annual-rate window used:", min(annual_month_cols), "to", max(annual_month_cols))

lsoa_crime = (
    lsoa_crime_raw
    .assign(Annual_Crime=lsoa_crime_raw[annual_month_cols].sum(axis=1))
    .groupby(["LSOA Code", "LSOA Name", "Borough"], as_index=False)["Annual_Crime"]
    .sum()
)

with TemporaryDirectory() as tmpdir:
    lookup_gpkg_path = Path(tmpdir) / "oa_lsoa_msoa_lookup.gpkg"
    urlretrieve(oa_lsoa_msoa_lookup_url, lookup_gpkg_path)
    with sqlite3.connect(lookup_gpkg_path) as con:
        lsoa_msoa_lookup = pd.read_sql_query(
            """
            SELECT DISTINCT LSOA21CD, LSOA21NM, MSOA21CD, MSOA21NM, LAD22CD, LAD22NM
            FROM OA21_LAD22_LSOA21_MSOA21_LEP22_EN_LU_V2
            """,
            con,
        )

lsoa_crime_mapped = lsoa_crime.merge(
    lsoa_msoa_lookup,
    left_on="LSOA Code",
    right_on="LSOA21CD",
    how="left",
)

lsoa_mapping_summary = pd.DataFrame({
    "metric": ["unique LSOAs in MPS crime", "LSOAs mapped to MSOA", "unmapped LSOAs"],
    "count": [
        lsoa_crime["LSOA Code"].nunique(),
        lsoa_crime_mapped.loc[lsoa_crime_mapped["MSOA21CD"].notna(), "LSOA Code"].nunique(),
        lsoa_crime_mapped.loc[lsoa_crime_mapped["MSOA21CD"].isna(), "LSOA Code"].nunique(),
    ],
})
display(lsoa_mapping_summary)

if lsoa_crime_mapped["MSOA21CD"].isna().any():
    print("Example unmapped LSOAs:")
    display(lsoa_crime_mapped[lsoa_crime_mapped["MSOA21CD"].isna()].head(10))

msoa_crime = (
    lsoa_crime_mapped
    .dropna(subset=["MSOA21CD"])
    .groupby(["MSOA21CD", "MSOA21NM", "LAD22CD", "LAD22NM"], as_index=False)
    .agg(
        Annual_Crime=("Annual_Crime", "sum"),
        LSOA_Count=("LSOA Code", "nunique"),
    )
)

print("MSOA crime data shape:", msoa_crime.shape)
print("London boroughs represented:", msoa_crime["LAD22CD"].nunique())

def load_census_table(key):
    df = pd.read_csv(census_urls[key])
    return df.rename(columns={"geography code": "MSOA21CD", "geography": "MSOA21NM_census"})

age = load_census_table("age")
deprivation = load_census_table("deprivation")
accommodation = load_census_table("accommodation")
tenure = load_census_table("tenure")
economic = load_census_table("economic")
qualification = load_census_table("qualification")

census_features = age[["MSOA21CD", "MSOA21NM_census", "Age: Total"]].copy()
census_features = census_features.rename(columns={"Age: Total": "Population_2021"})

age_child_cols = [
    "Age: Aged 4 years and under",
    "Age: Aged 5 to 9 years",
    "Age: Aged 10 to 14 years",
]
age_young_adult_cols = [
    "Age: Aged 15 to 19 years",
    "Age: Aged 20 to 24 years",
    "Age: Aged 25 to 29 years",
]
age_older_cols = [
    "Age: Aged 65 to 69 years",
    "Age: Aged 70 to 74 years",
    "Age: Aged 75 to 79 years",
    "Age: Aged 80 to 84 years",
    "Age: Aged 85 years and over",
]
census_features["Pct_Children_0_14"] = age[age_child_cols].sum(axis=1) / age["Age: Total"] * 100
census_features["Pct_Young_Adults_15_29"] = age[age_young_adult_cols].sum(axis=1) / age["Age: Total"] * 100
census_features["Pct_Older_65_plus"] = age[age_older_cols].sum(axis=1) / age["Age: Total"] * 100

hh_total = deprivation["Household deprivation: Total: All households; measures: Value"]
census_features["Pct_Households_Deprived_2plus"] = (
    deprivation[
        [
            "Household deprivation: Household is deprived in two dimensions; measures: Value",
            "Household deprivation: Household is deprived in three dimensions; measures: Value",
            "Household deprivation: Household is deprived in four dimensions; measures: Value",
        ]
    ].sum(axis=1) / hh_total * 100
)

accom_total = accommodation["Accommodation type: Total: All households"]
census_features["Pct_Flats"] = (
    accommodation[
        [
            "Accommodation type: In a purpose-built block of flats or tenement",
            "Accommodation type: Part of a converted or shared house, including bedsits",
            "Accommodation type: Part of another converted building, for example, former school, church or warehouse",
            "Accommodation type: In a commercial building, for example, in an office building, hotel or over a shop",
        ]
    ].sum(axis=1) / accom_total * 100
)

tenure_total = tenure["Tenure of household: Total: All households"]
census_features["Pct_Social_Rented"] = tenure["Tenure of household: Social rented"] / tenure_total * 100
census_features["Pct_Private_Rented"] = tenure["Tenure of household: Private rented"] / tenure_total * 100
census_features["Pct_Owner_Occupied"] = tenure["Tenure of household: Owned"] / tenure_total * 100

econ_total = economic["Economic activity status: Total: All usual residents aged 16 years and over"]
census_features["Pct_Unemployed"] = economic["Economic activity status: Economically active (excluding full-time students): Unemployed"] / econ_total * 100
census_features["Pct_Inactive_Long_Term_Sick"] = economic["Economic activity status: Economically inactive: Long-term sick or disabled"] / econ_total * 100

qual_total = qualification["Highest level of qualification: Total: All usual residents aged 16 years and over"]
census_features["Pct_No_Qualifications"] = qualification["Highest level of qualification: No qualifications"] / qual_total * 100
census_features["Pct_Level4_Qualifications"] = qualification["Highest level of qualification: Level 4 qualifications and above"] / qual_total * 100

msoa_df = msoa_crime.merge(census_features, on="MSOA21CD", how="inner")
print(f"MSOA crime-census matched rows: {len(msoa_df)} out of {msoa_crime['MSOA21CD'].nunique()} crime MSOAs")
display(msoa_df.head())


LSOA crime data shape: (127288, 173)
Number of monthly crime columns in source: 168
Source date range: 201004 to 202403
Annual-rate window used: 202101 to 202112


,metric,count
0,unique LSOAs in MPS crime,4988
1,LSOAs mapped to MSOA,4988
2,unmapped LSOAs,0


MSOA crime data shape: (1001, 6)
London boroughs represented: 32
MSOA crime-census matched rows: 1001 out of 1001 crime MSOAs


,MSOA21CD,MSOA21NM,LAD22CD,LAD22NM,Annual_Crime,LSOA_Count,MSOA21NM_census,Population_2021,Pct_Children_0_14,Pct_Young_Adults_15_29,Pct_Older_65_plus,Pct_Households_Deprived_2plus,Pct_Flats,Pct_Social_Rented,Pct_Private_Rented,Pct_Owner_Occupied,Pct_Unemployed,Pct_Inactive_Long_Term_Sick,Pct_No_Qualifications,Pct_Level4_Qualifications
0,E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham,627,4,Barking and Dagenham 001,8286,26.212889,18.657977,11.211682,32.607749,39.930314,41.777003,14.529617,43.066202,4.661654,5.664160,23.943191,30.910610
1,E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham,982,6,Barking and Dagenham 002,11539,22.185631,20.071063,11.014819,30.262786,32.374650,13.282443,30.458015,55.623410,4.341927,2.883311,21.301842,34.015143
2,E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham,382,4,Barking and Dagenham 003,6638,18.650196,19.674601,15.049714,31.522075,13.902122,16.154179,16.284106,66.825466,4.214487,3.631232,23.499530,30.743180
3,E02000005,Barking and Dagenham 004,E09000002,Barking and Dagenham,696,6,Barking and Dagenham 004,11082,26.484389,18.453348,7.967876,29.138322,14.112459,25.137817,21.471885,51.874311,4.044072,4.194316,22.268119,33.783953
4,E02000007,Barking and Dagenham 006,E09000002,Barking and Dagenham,957,5,Barking and Dagenham 006,10161,24.859758,20.086606,9.615195,30.522946,40.902910,42.219804,16.376496,39.526659,4.391485,5.342081,23.801232,31.703724


**English**

The selected variables are designed to be readable and theoretically motivated. They cover population scale, age structure, deprivation, housing form, tenure, labour-market position and education. The dependent variable, Annual_Crime_Rate, the binary label, High_Risk, and the aggregation check variable, LSOA_Count, are not included in the feature matrix.

**中文翻译**

所选变量力求清晰且有理论依据，覆盖人口规模、年龄结构、贫困、住房形态、tenure、劳动力市场状态和教育。因变量 Annual_Crime_Rate、二分类标签 High_Risk 和聚合检查变量 LSOA_Count 不进入特征矩阵。




| Variable | Type | Description | Notes |
|---|---|---|---|
| Annual_Crime_Rate | Numeric | 2021 recorded crime rate per 1,000 residents at MSOA level | Used to construct Y, not used as X |
| High_Risk | Binary | 1 if MSOA is in top 25% of Annual_Crime_Rate, otherwise 0 | Dependent variable Y |
| Population_2021 | Numeric | Census 2021 usual resident population | Predictor X |
| Pct_Households_Deprived_2plus | Numeric | Share of households deprived in two or more dimensions | Predictor X |
| Pct_Flats | Numeric | Share of households in flats or converted/shared accommodation | Predictor X |
| Pct_Social_Rented / Pct_Private_Rented | Numeric | Tenure structure | Predictor X |
| Pct_Unemployed | Numeric | Unemployed share among residents aged 16+ | Predictor X |
| Pct_No_Qualifications / Pct_Level4_Qualifications | Numeric | Education profile | Predictor X |

**中文翻译：** 该表概括了主要变量。完整变量字典会由下面代码自动生成。


## Methodology

[[ go back to the top ]](#Table-of-contents)

**English**

The workflow has four main stages. First, I aggregate 2021 LSOA recorded-crime counts to MSOA level using the ONS lookup. Second, I calculate the annual crime rate per 1,000 residents using Census 2021 population and define the top quartile as the high-risk class. Third, I merge this outcome with selected Census 2021 features. Fourth, I train and evaluate two classification models.

The first model is Logistic Regression. It is used as a baseline because it is simple and its coefficients can be interpreted as positive or negative associations with high-risk status. The second model is a Random Forest classifier. It is used to test whether a more flexible model can capture non-linear patterns that the linear baseline may miss. Both models use class_weight='balanced' because the high-risk group is smaller than the low-risk group.

The data are split into training and test sets using stratification, so that both sets keep a similar class balance. I also use 5-fold StratifiedKFold cross-validation to reduce dependence on a single random split. The main metrics are precision, recall and F1-score for the high-risk class. Recall is especially important here because a false negative means a high-risk MSOA is missed by the model.

**中文翻译**

分析流程分为四个主要阶段。第一，将 2021 年 LSOA 记录犯罪数通过 ONS lookup 聚合到 MSOA 层级。第二，用 Census 2021 人口计算每 1,000 名居民的年度犯罪率，并将前 25% 定义为高风险类别。第三，将该目标变量与选定的 Census 2021 特征合并。第四，训练并评估两个分类模型。

第一个模型是 Logistic Regression。它作为基线模型，因为结构简单，系数可以解释为与高风险状态的正向或负向关联。第二个模型是 Random Forest 分类器，用来检验更灵活的模型是否能捕捉线性基线可能遗漏的非线性模式。两个模型都使用 class_weight='balanced'，因为高风险类别比低风险类别更少。

数据使用 stratified train-test split，使训练集和测试集保持相似的类别比例。本研究还使用 5-fold StratifiedKFold 交叉验证，减少单次随机划分的影响。主要指标是 high-risk 类别的 precision、recall 和 F1-score。这里尤其重视 recall，因为 false negative 表示模型漏掉了一个高风险 MSOA。




**Methodology flow chart**

![Research flow chart](methodology_flow_chart.svg)

**Figure 1.** Research flow chart showing how the crime data, lookup table, Census features and boundary data are combined before model evaluation and interpretation.

**中文翻译：** 图 1 展示了本研究的整体流程：犯罪数据、lookup、Census 特征和边界数据如何被整合，并进一步用于模型评估、地图和结果解释。




In [20]:
# 2. 目标构建 / Target engineering: build MSOA crime rates and binary labels.
# 年犯罪率 / Annual crime rate = MPS 2021 recorded crimes / Census 2021 MSOA population * 1,000.
analysis_df = msoa_df.copy()
analysis_df = analysis_df.dropna(subset=["Annual_Crime", "Population_2021"]).copy()
analysis_df["Annual_Crime_Rate"] = (analysis_df["Annual_Crime"] / analysis_df["Population_2021"]) * 1000

# 二值标签 / Binary label: mark the top 25% of 2021 MSOA crime rates as high risk.
crime_rate_threshold = analysis_df["Annual_Crime_Rate"].quantile(0.75)
analysis_df["High_Risk"] = (analysis_df["Annual_Crime_Rate"] >= crime_rate_threshold).astype(int)

print(f"75th percentile annual crime-rate threshold: {crime_rate_threshold:.2f} crimes per 1,000 residents")
print("Class balance among MSOAs:")
display(
    analysis_df["High_Risk"]
    .value_counts()
    .rename(index={0: "Low risk", 1: "High risk"})
    .to_frame("count")
    .assign(share=lambda x: x["count"] / x["count"].sum())
)

display(
    analysis_df[["MSOA21CD", "MSOA21NM", "LAD22NM", "Annual_Crime", "Population_2021", "Annual_Crime_Rate", "High_Risk"]]
    .sort_values("Annual_Crime_Rate", ascending=False)
    .head(10)
)


75th percentile annual crime-rate threshold: 94.32 crimes per 1,000 residents
Class balance among MSOAs:


,count,share
High_Risk,,
Low risk,750,0.749251
High risk,251,0.250749


,MSOA21CD,MSOA21NM,LAD22NM,Annual_Crime,Population_2021,Annual_Crime_Rate,High_Risk
924,E02000977,Westminster 018,Westminster,10489,6389,1641.727970,1
919,E02000972,Westminster 013,Westminster,12071,7875,1532.825397,1
182,E02000193,Camden 028,Camden,3334,5958,559.583753,1
996,E02007111,Hackney 033,Hackney,2800,5952,470.430108,1
917,E02000970,Westminster 011,Westminster,3130,6832,458.138173,1
180,E02000191,Camden 026,Camden,2283,5340,427.528090,1
948,E02006801,Lambeth 036,Lambeth,3167,8565,369.760654,1
177,E02000186,Camden 021,Camden,2178,6224,349.935733,1
981,E02006996,Newham 039,Newham,2949,8429,349.863566,1
385,E02000412,Haringey 016,Haringey,2477,7326,338.110838,1


In [ ]:
# 3. EDA / Exploratory analysis: inspect target distribution and selected variables.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=analysis_df, x="Annual_Crime_Rate", hue="High_Risk", bins=30, ax=axes[0])
axes[0].axvline(crime_rate_threshold, color="black", linestyle="--", label="75th percentile")
axes[0].set_title("MSOA 2021 recorded crime-rate distribution")
axes[0].set_xlabel("Annual crime rate per 1,000 residents")
axes[0].legend()

sns.countplot(data=analysis_df, x="High_Risk", ax=axes[1])
axes[1].set_title("Binary target balance")
axes[1].set_xlabel("High risk label")
axes[1].set_xticklabels(["Low risk (0)", "High risk (1)"])
axes[1].set_ylabel("Number of MSOAs")

plt.tight_layout()
plt.show()

selected_feature_candidates = [
    "Population_2021",
    "Pct_Children_0_14",
    "Pct_Young_Adults_15_29",
    "Pct_Older_65_plus",
    "Pct_Households_Deprived_2plus",
    "Pct_Flats",
    "Pct_Social_Rented",
    "Pct_Private_Rented",
    "Pct_Owner_Occupied",
    "Pct_Unemployed",
    "Pct_Inactive_Long_Term_Sick",
    "Pct_No_Qualifications",
    "Pct_Level4_Qualifications",
]
selected_feature_candidates = [col for col in selected_feature_candidates if col in analysis_df.columns]

variable_dictionary = pd.DataFrame({
    "Variable": selected_feature_candidates,
    "Type": [str(analysis_df[col].dtype) for col in selected_feature_candidates],
    "Missing values": [int(analysis_df[col].isna().sum()) for col in selected_feature_candidates],
    "Description / Notes": "Selected Census 2021 MSOA feature; aggregation count excluded",
})
print(f"Selected modelling features: {len(selected_feature_candidates)}")
display(variable_dictionary)



In [ ]:
# 4. 空间可视化 / Spatial visualisation: choropleth map of recorded-crime risk.
# 边界数据 / Boundary data: read the MSOA 2021 GPKG from GitHub and join by MSOA21CD.
try:
    import geopandas as gpd

    with TemporaryDirectory() as tmpdir:
        boundary_gpkg_path = Path(tmpdir) / "msoa_boundary.gpkg"
        urlretrieve(msoa_boundary_url, boundary_gpkg_path)
        msoa_gdf = gpd.read_file(boundary_gpkg_path)

    code_col = "MSOA21CD"
    boundary_codes = set(msoa_gdf[code_col])
    analysis_codes = set(analysis_df["MSOA21CD"])
    boundary_matches = len(boundary_codes & analysis_codes)
    print(f"MSOA boundary areas matching analysis data: {boundary_matches} out of {len(analysis_codes)} analysis MSOAs")

    map_gdf = msoa_gdf[msoa_gdf[code_col].isin(analysis_codes)].merge(
        analysis_df[["MSOA21CD", "MSOA21NM", "LAD22NM", "Annual_Crime_Rate", "High_Risk"]],
        on="MSOA21CD",
        how="left",
    )
    mapped_label_count = int(map_gdf["High_Risk"].notna().sum())
    print(f"Map join labels available for: {mapped_label_count} out of {len(map_gdf)} displayed MSOAs")

    valid_rate = map_gdf["Annual_Crime_Rate"].notna()
    map_gdf.loc[valid_rate, "Risk_Band"] = pd.qcut(
        map_gdf.loc[valid_rate, "Annual_Crime_Rate"],
        q=4,
        labels=False,
        duplicates="drop",
    )

    palette = {
        0: "#f2f2f2",
        1: "#fee8c8",
        2: "#fdbb84",
        3: "#d7301f",
    }
    band_names = {
        0: "Lowest risk quartile",
        1: "Lower-middle risk quartile",
        2: "Upper-middle risk quartile",
        3: "High risk (top 25%)",
    }

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    missing_risk = map_gdf[map_gdf["Risk_Band"].isna()]

    for band in sorted(map_gdf["Risk_Band"].dropna().unique()):
        band = int(band)
        band_gdf = map_gdf[map_gdf["Risk_Band"] == band]
        band_gdf.plot(
            color=palette.get(band, "#d7301f"),
            linewidth=0.12,
            edgecolor="white",
            ax=ax,
        )

    if len(missing_risk) > 0:
        missing_risk.plot(color="#bdbdbd", linewidth=0.12, edgecolor="white", ax=ax)

    handles = []
    for band in sorted(map_gdf["Risk_Band"].dropna().unique()):
        band = int(band)
        band_rates = map_gdf.loc[map_gdf["Risk_Band"] == band, "Annual_Crime_Rate"]
        label = f"{band_names.get(band, 'Risk band')} ({band_rates.min():.1f}-{band_rates.max():.1f} per 1,000)"
        handles.append(Patch(facecolor=palette.get(band, "#d7301f"), edgecolor="white", label=label))
    if len(missing_risk) > 0:
        handles.append(Patch(facecolor="#bdbdbd", edgecolor="white", label="No data"))

    ax.legend(
        handles=handles,
        loc="lower left",
        frameon=True,
        title="Recorded-crime risk",
        fontsize=8,
        title_fontsize=9,
    )
    ax.set_title("Recorded-crime risk by London MSOA, 2021")
    ax.text(
        0.01,
        0.01,
        "Red indicates the highest-risk MSOAs (top quartile of recorded-crime rate).",
        transform=ax.transAxes,
        fontsize=8,
        color="#333333",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 3},
    )
    ax.axis("off")
    plt.show()
except Exception as err:
    print("Spatial map skipped because geopandas/MSOA boundary reading is unavailable:", err)



In [ ]:
# 5. 建模准备 / Modelling setup.
# 特征矩阵 / Feature matrix: exclude IDs, names, target variables and leakage columns.
X = analysis_df[selected_feature_candidates].copy()
y = analysis_df["High_Risk"].copy()

X = X.dropna(axis=1, how="all")

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()

print("Number of selected features:", X.shape[1])
print("Numeric features:", len(numeric_features))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train class balance:")
print(y_train.value_counts(normalize=True).sort_index())

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
])


In [ ]:
# 6. 基线模型 / Baseline model: Logistic Regression.
# 类别权重 / Class weighting handles the roughly 1:3 class imbalance; coefficients remain interpretable.
log_reg_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

log_reg_model.fit(X_train, y_train)
log_reg_pred = log_reg_model.predict(X_test)

print("Logistic Regression classification report:")
print(classification_report(y_test, log_reg_pred, target_names=["Low risk", "High risk"]))


In [ ]:
# 7. 进阶模型 / Advanced model: Random Forest classifier.
# 随机森林 / Random Forest captures non-linear patterns; balanced weights support the minority high-risk class.
rf_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest classification report:")
print(classification_report(y_test, rf_pred, target_names=["Low risk", "High risk"]))


## Results and discussion

[[ go back to the top ]](#Table-of-contents)

**English**

The data preparation results show why the final MSOA design was preferable to my initial ward design. At the start, I expected wards to be the most suitable unit because they are familiar local policy areas and the Metropolitan Police provide ward-level crime tables. I therefore tried to combine ward crime with the ward profile dataset. The problem was that the ward crime codes and the ward profile geography did not match consistently. Some areas had changed boundaries or identifiers, so a ward-level join would have mixed incompatible spatial units and could have made the model look more reliable than it really was.

This mismatch changed the direction of the project. Rather than forcing the ward datasets together, I moved to LSOA crime data aggregated into MSOAs. This was a more defensible workflow because all 4,988 LSOAs in the crime table are mapped to MSOAs, and the resulting 1,001 London MSOAs all match the Census feature table. The final target also has the intended class balance: 251 MSOAs are labelled high risk and 750 are labelled low risk. This gives a clear but imbalanced classification task, so recall needs to be read alongside precision.

The model comparison is read through the high-risk class rather than overall accuracy alone. The single test split and the 5-fold cross-validation results are used together to check whether one model is consistently better at identifying high-risk MSOAs. Because the project is framed as a screening task, missing high-risk areas is the more serious error. The preferred model is therefore the one with stronger high-risk recall, while precision and F1-score show the cost of that screening approach.

The interpretation results also make the model easier to discuss. The coefficient and feature-importance plots now focus only on Census socio-demographic and housing variables, rather than technical aggregation fields. If tenure, housing form or age structure appear among the strongest predictors, this suggests that area composition provides useful signals for distinguishing high-risk MSOAs. These results are associations rather than causes.

Several limitations remain. Recorded crime depends on reporting and policing practices, so it is not the same as all crime that occurred. MSOAs are useful for analysis, but they still hide variation within areas. The year 2021 was also affected by pandemic conditions, so crime patterns may not fully represent normal urban activity. The final model is therefore an exploratory classification tool, not an automatic decision system.

**中文翻译**

数据准备结果说明了为什么最终的 MSOA 设计比我最初的 ward 设计更合适。一开始，我认为 ward 可能是最合适的分析单元，因为 ward 是常见的地方政策区域，而且 Metropolitan Police 也提供 ward 层级犯罪表。因此我先尝试把 ward crime 和 ward profile 数据结合起来。问题在于，ward 犯罪编码和 ward profile 的地理边界并不能稳定匹配。有些区域的边界或 identifier 已经变化，所以如果强行做 ward 层级 join，就会混合不兼容的空间单元，并可能让模型看起来比实际更可靠。

这个不匹配问题改变了项目方向。我没有继续强行拼接 ward 数据，而是改用 LSOA crime 数据并聚合到 MSOA。这是更稳妥的流程，因为 crime 表中的 4,988 个 LSOA 都能映射到 MSOA，最终得到的 1,001 个伦敦 MSOA 也都能匹配 Census 特征。最终目标变量也形成了预期的类别比例：251 个 MSOA 被标记为高风险，750 个被标记为低风险。这使任务清晰，但也存在类别不平衡，因此 recall 需要和 precision 一起理解。

模型比较重点查看 high-risk 类别，而不是只看整体 accuracy。单次测试集和 5-fold 交叉验证结果一起使用，以判断哪个模型能更稳定地识别高风险 MSOA。由于本项目被设定为筛查任务，漏掉高风险区域是更严重的错误。因此，首选模型是 high-risk recall 更强的模型，而 precision 和 F1-score 则显示这种筛查取向的代价。

模型解释结果也让讨论更清楚。系数图和特征重要性图现在只关注 Census 社会人口和住房变量，而不是技术性的聚合字段。如果 tenure、住房形态或年龄结构出现在最重要变量中，说明区域构成可以为区分高风险 MSOA 提供有用信号。不过，这些结果是关联，而不是因果关系。

本研究仍有局限。记录犯罪受到报案行为和警务实践影响，因此不等于实际发生的全部犯罪。MSOA 适合分析，但仍会掩盖区域内部差异。2021 年还受到疫情条件影响，因此犯罪模式不一定完全代表正常城市活动。最终模型是探索性分类工具，而不是自动决策系统。






In [ ]:
# 8. 模型评估 / Model evaluation and comparison.
# 评价重点 / Evaluation focus: high-risk recall matters because false negatives weaken preventive resource allocation.
def collect_metrics(name, y_true, y_pred):
    return {
        "Model": name,
        "Precision_high_risk": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall_high_risk": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1_high_risk": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

results_df = pd.DataFrame([
    collect_metrics("Logistic Regression", y_test, log_reg_pred),
    collect_metrics("Random Forest", y_test, rf_pred),
]).sort_values("Recall_high_risk", ascending=False)

print("Single stratified train-test split performance:")
display(results_df)

# 交叉验证 / Cross-validation: StratifiedKFold reduces dependence on one random train-test split.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scoring = {
    "precision_high_risk": "precision",
    "recall_high_risk": "recall",
    "f1_high_risk": "f1",
}

cv_rows = []
for name, model in [
    ("Logistic Regression", log_reg_model),
    ("Random Forest", rf_model),
]:
    cv_scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=cv_scoring,
        n_jobs=-1,
        error_score="raise",
    )
    cv_rows.append({
        "Model": name,
        "CV_precision_high_risk_mean": cv_scores["test_precision_high_risk"].mean(),
        "CV_precision_high_risk_std": cv_scores["test_precision_high_risk"].std(),
        "CV_recall_high_risk_mean": cv_scores["test_recall_high_risk"].mean(),
        "CV_recall_high_risk_std": cv_scores["test_recall_high_risk"].std(),
        "CV_f1_high_risk_mean": cv_scores["test_f1_high_risk"].mean(),
        "CV_f1_high_risk_std": cv_scores["test_f1_high_risk"].std(),
    })

cv_results_df = pd.DataFrame(cv_rows).sort_values("CV_recall_high_risk_mean", ascending=False)
print("5-fold StratifiedKFold cross-validation performance:")
display(cv_results_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name, pred in zip(
    axes,
    ["Logistic Regression", "Random Forest"],
    [log_reg_pred, rf_pred],
):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        pred,
        display_labels=["Low risk", "High risk"],
        cmap="Blues",
        ax=ax,
        colorbar=False,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()


In [ ]:
# 9. 模型解释 / Model interpretation: logistic coefficients and Random Forest feature importance.
# 特征名 / Feature names: recover transformed feature names for interpretation.
feature_names = log_reg_model.named_steps["preprocess"].get_feature_names_out()
clean_feature_names = [name.replace("num__", "").replace("cat__", "") for name in feature_names]

coef_df = pd.DataFrame({
    "feature": clean_feature_names,
    "coefficient": log_reg_model.named_steps["model"].coef_[0],
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_top = coef_df.sort_values("abs_coefficient", ascending=False).head(12)

display(coef_top)

plt.figure(figsize=(10, 6))
sns.barplot(data=coef_top, y="feature", x="coefficient", palette="vlag")
plt.axvline(0, color="black", linewidth=1)
plt.title("Largest logistic regression coefficients")
plt.xlabel("Coefficient: positive values increase high-risk probability")
plt.ylabel("")
plt.tight_layout()
plt.show()

rf_importance_df = pd.DataFrame({
    "feature": clean_feature_names,
    "importance": rf_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False)
rf_top = rf_importance_df.head(12)

display(rf_top)

plt.figure(figsize=(10, 6))
sns.barplot(data=rf_top, y="feature", x="importance", color="#4C78A8")
plt.title("Random Forest feature importance")
plt.xlabel("Mean decrease in impurity")
plt.ylabel("")
plt.tight_layout()
plt.show()


## Conclusion

[[ go back to the top ]](#Table-of-contents)

**English**

This study shows that 2021 Census characteristics can distinguish high recorded-crime risk MSOAs in London to a useful but limited extent. The MSOA design gives a consistent geography across crime, Census features and boundary data. The final model comparison is based on cross-validated high-risk recall and F1-score. The model is useful as a screening approach, but it is not accurate enough for automatic decisions.

The analysis is a classification and interpretation exercise, not a causal model. It can help show which types of areas are associated with higher recorded-crime risk, especially through housing and tenure variables, but it is not suitable for labelling neighbourhoods without further local evidence. Future work could add transport accessibility, land use, night-time economy, points of interest and multi-year crime trends.

**中文翻译**

本研究表明，2021 Census 特征可以在一定程度上区分伦敦高记录犯罪风险 MSOA，但这种能力是有用而有限的。MSOA 方案使犯罪数据、Census 特征和边界数据保持一致的地理尺度。最终模型比较基于交叉验证 high-risk recall 和 F1-score。模型可以作为筛查方法，但还不足以用于自动决策。

本分析是分类和解释练习，而不是因果模型。它可以帮助说明哪些类型的地区与更高记录犯罪风险相关，尤其是住房和 tenure 变量，但不适合在缺少进一步地方证据的情况下直接给社区贴标签。未来研究可以加入交通可达性、土地利用、夜间经济、兴趣点和多年犯罪趋势。






In [ ]:
# 10. 结果摘要 / Auto-generate a concise summary for Results and Conclusion writing.
best_model_row = cv_results_df.iloc[0]
print("Model comparison summary")
print("------------------------")
print(f"Best model by cross-validated high-risk recall: {best_model_row['Model']}")
print(f"CV high-risk recall: {best_model_row['CV_recall_high_risk_mean']:.3f} ± {best_model_row['CV_recall_high_risk_std']:.3f}")
print(f"CV high-risk precision: {best_model_row['CV_precision_high_risk_mean']:.3f} ± {best_model_row['CV_precision_high_risk_std']:.3f}")
print(f"CV high-risk F1-score: {best_model_row['CV_f1_high_risk_mean']:.3f} ± {best_model_row['CV_f1_high_risk_std']:.3f}")
print()
print("Interpretation note:")
print(
    "In this resource-allocation framing, false negatives are especially costly because "
    "they represent high-risk MSOAs that the model fails to identify. Therefore recall "
    "for the High risk class should be discussed alongside precision and F1-score. "
    "The modelling evidence applies to MSOAs with matched LSOA crime, Census 2021 features, and MSOA 2021 geography."
)



## References

[[ go back to the top ]](#Table-of-contents)

**English**

Brantingham, P. L. and Brantingham, P. J. (1993) 'Environment, routine and situation: Toward a pattern theory of crime', in Clarke, R. V. and Felson, M. (eds.) *Routine Activity and Rational Choice*. New Brunswick: Transaction, pp. 259-294.

Bursik, R. J. and Grasmick, H. G. (1993) *Neighborhoods and Crime: The Dimensions of Effective Community Control*. New York: Lexington Books.

Metropolitan Police Service (2025) *LSOA Level Crime Data*. Available via London Datastore / MPS recorded crime data.

Office for National Statistics (2021) *Census 2021 MSOA tables, Census geography lookup and MSOA boundary data*. London: ONS.

Sampson, R. J., Raudenbush, S. W. and Earls, F. (1997) 'Neighborhoods and violent crime: A multilevel study of collective efficacy', *Science*, 277(5328), pp. 918-924. doi:10.1126/science.277.5328.918.

scikit-learn developers (2025) *scikit-learn documentation: LogisticRegression, RandomForestClassifier, classification_report and StratifiedKFold*.

**中文翻译**

以上参考文献包括四类：社区犯罪与环境犯罪学理论、MPS/ONS 数据来源，以及 scikit-learn 方法文档。参考文献按统一格式列出，并优先保留与研究背景、数据和方法直接相关的来源。





In [29]:
# 参考文献提示 / Reference checklist:
# 1. 在正文中引用 Metropolitan Police Service LSOA-level crime data。
# 2. 引用 ONS Census 2021 MSOA tables used for socio-demographic and housing features。
# 3. 引用 ONS OA/LSOA/MSOA lookup and MSOA 2021 boundary data。
# 4. 引用 scikit-learn 文档中 LogisticRegression、RandomForestClassifier、classification_report 等方法。
# This reminder cell does not need to be executed.
